# 2.3 推理加速策略

## 什么是推理加速？

在不改变模型参数的前提下，通过优化推理过程中的计算策略（如投机解码、批量优化、早期退出等），显著提升推理速度和降低延迟。

## 为什么需要推理加速？

1. **自回归瓶颈**：LLM逐token生成，每步都需要完整前向传播，GPU利用率极低（通常<5%）。
2. **内存带宽受限**：推理的瓶颈往往是内存带宽而非计算，每次前向传播需要加载全部模型权重。
3. **延迟敏感**：端侧应用（对话、翻译）对首token延迟和吞吐量都有严格要求。

### 直观理解：为什么LLM推理这么慢？

想象你在写一篇文章，每次只能写一个字，写每个字之前都要重新读一遍前面写过的所有内容——这就是LLM自回归推理的困境。虽然GPU有上万个计算核心，但生成一个token时，大部分核心都在等待，因为计算量太小而加载权重的开销太大。

```
自回归生成过程（每步只产出1个token）:

Step 1: [输入] → 模型前向传播 → token_1    ← 加载全部权重，计算量很小
Step 2: [输入, token_1] → 模型前向传播 → token_2  ← 又加载全部权重
Step 3: [输入, token_1, token_2] → 模型前向传播 → token_3  ← 又加载全部权重
...每次都要加载全部权重，但只用了很少的计算能力
```

## 推理加速的核心思路

| 思路 | 方法 | 核心思想 |
|------|------|----------|
| 减少前向传播次数 | 投机解码 / Medusa | 一次验证多个token |
| 提高GPU利用率 | 连续批处理 | 减少GPU空闲 |
| 减少计算量 | 早期退出 | 跳过不必要的层 |
| 减少重复计算 | 前缀缓存 | 复用公共前缀的KV Cache |
| 优化Prefill阶段 | 分块预填充 | 避免长输入导致的延迟尖峰 |

## 本章内容导航

```
2.3 推理加速策略
├── 2.3.1 投机解码（Speculative Decoding）
│   ├── 标准投机解码：小模型生成，大模型验证
│   └── 自投机解码：浅层生成，深层验证
├── 2.3.2 Medusa多头预测
├── 2.3.3 连续批处理（Continuous Batching）
├── 2.3.4 早期退出（Early Exit）
├── 2.3.5 前缀缓存与分块预填充
├── 方法综合对比
├── 产业级实战：真实模型投机解码
└── 总结与部署建议
```

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import time
import gc
from typing import Optional, Tuple, List, Dict

torch.manual_seed(42)
np.random.seed(42)
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

---
## 2.3.1 投机解码（Speculative Decoding）

### 什么是投机解码？

使用小模型（draft model）快速生成多个候选token，再用大模型（target model）并行验证这些token，接受正确的token、拒绝错误的token。在保持与大模型完全相同输出分布的前提下，显著提升推理速度。

### 直观理解：投机解码就像"先写草稿，再批改"

```
传统自回归（逐字写，每字都请教授确认）:
  你 → 教授确认 → 好 → 教授确认 → 吗 → 教授确认 → ？ → 教授确认
  每次确认都要教授（大模型）花时间

投机解码（学生先写草稿，教授一次性批改）:
  学生草稿: 你 好 吗 ？ 今 天
  教授批改: ✓  ✓  ✓  ✗  
  结果: 你 好 吗 [教授修正] 天气
  
  学生写5个字很快，教授批改5个字只需1次审阅
  如果前3个字都对了，相当于1次审阅就确认了3+1=4个字！
```

### 为什么投机解码有效？

1. **利用GPU并行性**：大模型验证$k$个token只需1次前向传播（与验证1个token相同），而小模型生成$k$个token的成本很低。
2. **高接受率**：如果draft模型与target模型分布接近（如同一系列的大小模型），大部分候选token会被接受，平均每步可接受2-4个token。
3. **分布不变**：通过拒绝采样保证输出分布与target模型完全一致，不损失任何精度。

### 投机解码的数学原理

**接受-拒绝采样**：对于draft模型生成的token $x$，

$$\text{accept} = \begin{cases} \text{True} & \text{w.p. } \min\left(1, \frac{p_{target}(x)}{p_{draft}(x)}\right) \\ \text{False} & \text{otherwise} \end{cases}$$

**拒绝后修正采样**：
$$p_{correct}(x) = \frac{\max(0, p_{target}(x) - p_{draft}(x))}{\sum_{x'} \max(0, p_{target}(x') - p_{draft}(x'))}$$

其中：
- $p_{target}(x)$：target模型对token $x$ 的概率
- $p_{draft}(x)$：draft模型对token $x$ 的概率
- 接受概率：$p_{draft}$和$p_{target}$越接近，接受率越高
- 修正分布：确保最终输出严格遵循$p_{target}$分布
- 理论加速比：$\frac{1}{1 - \alpha}$，其中$\alpha$为平均接受率

### 投机解码的执行流程

```
输入: prompt = [A, B, C]

Step 1: Draft模型快速生成k=4个候选token
  [A, B, C] → Draft → D, E, F, G
  得到候选序列: [A, B, C, D, E, F, G]

Step 2: Target模型一次前向传播验证所有候选
  [A, B, C, D, E, F, G] → Target → 每个位置的概率分布
  对比每个候选token:
    位置3: p_target(D) vs p_draft(D) → 接受 ✓
    位置4: p_target(E) vs p_draft(E) → 接受 ✓
    位置5: p_target(F) vs p_draft(F) → 拒绝 ✗ → 从修正分布采样F'

Step 3: 输出
  接受的tokens: [D, E] + 修正token [F']
  本轮产出3个token，但Target只做了1次前向传播！
  如果全部接受，还能额外获得1个bonus token
```

In [ ]:
class SimpleLM(nn.Module):
    """简化语言模型，用于演示投机解码

    实际应用中，draft和target模型通常是:
    - target: Llama-70B, GPT-2 medium 等
    - draft: Llama-7B, GPT-2 small 等（同系列的小模型）
    """
    def __init__(self, vocab_size: int = 100, dim: int = 64, n_layers: int = 2):
        super().__init__()
        self.vocab_size = vocab_size
        self.emb = nn.Embedding(vocab_size, dim)
        layers: list = []
        for _ in range(n_layers):
            layers.append(nn.Linear(dim, dim))
            layers.append(nn.ReLU())
        self.net = nn.Sequential(*layers)
        self.head = nn.Linear(dim, vocab_size)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h = self.emb(x)
        h = self.net(h)
        return self.head(h)

vocab_size = 100
draft_model = SimpleLM(vocab_size, dim=32, n_layers=1)
target_model = SimpleLM(vocab_size, dim=64, n_layers=3)

draft_params = sum(p.numel() for p in draft_model.parameters())
target_params = sum(p.numel() for p in target_model.parameters())
print(f"Draft模型参数量: {draft_params:,}")
print(f"Target模型参数量: {target_params:,}")
print(f"参数比: draft/target = {draft_params/target_params:.1%}")

In [ ]:
def speculative_decode(
    draft: nn.Module,
    target: nn.Module,
    prompt: torch.Tensor,
    max_new_tokens: int = 20,
    draft_len: int = 5,
    temperature: float = 1.0,
) -> Tuple[torch.Tensor, float, float, int]:
    """投机解码实现

    Args:
        draft: 小模型，快速生成draft_len个候选token
        target: 大模型，并行验证候选token
        prompt: 输入token序列，shape=(seq_len,)
        max_new_tokens: 最大生成token数
        draft_len: 每轮draft模型生成的候选token数
        temperature: 采样温度，1.0为标准采样

    Returns:
        (generated, acceptance_rate, speedup, n_steps)
        - generated: 生成的完整序列（含prompt）
        - acceptance_rate: draft token被target接受的比例
        - speedup: 相比自回归的理论加速比
        - n_steps: target模型前向传播次数
    """
    device = next(target.parameters()).device
    generated = prompt.clone().to(device)
    prompt_len = prompt.shape[0]
    total_draft = 0
    total_accepted = 0
    n_steps = 0

    while generated.shape[0] - prompt_len < max_new_tokens:
        n_steps += 1
        draft_tokens: list = []
        draft_probs_list: list = []
        current = generated.clone()

        remaining = max_new_tokens - (generated.shape[0] - prompt_len)
        k = min(draft_len, remaining)

        with torch.no_grad():
            for _ in range(k):
                logits = draft(current.unsqueeze(0))
                probs = F.softmax(logits[0, -1] / max(temperature, 1e-8), dim=-1)
                token = torch.multinomial(probs, num_samples=1).squeeze()
                draft_tokens.append(token)
                draft_probs_list.append(probs)
                current = torch.cat([current, token.unsqueeze(0)])

        total_draft += len(draft_tokens)

        with torch.no_grad():
            target_logits = target(current.unsqueeze(0))
            target_all_probs = F.softmax(target_logits[0] / max(temperature, 1e-8), dim=-1)

        n_accepted = 0
        for i, (dt, dp) in enumerate(zip(draft_tokens, draft_probs_list)):
            pos = prompt_len - 1 + i

            tp = target_all_probs[pos]
            accept_prob = torch.min(
                tp[dt] / (dp[dt] + 1e-10),
                torch.tensor(1.0, device=device),
            )

            if torch.rand(1, device=device).item() < accept_prob.item():
                n_accepted += 1
                generated = torch.cat([generated, dt.unsqueeze(0)])
            else:
                diff = torch.clamp(tp - dp, min=0)
                if diff.sum() > 0:
                    corrected = torch.multinomial(diff / diff.sum(), num_samples=1).squeeze()
                else:
                    corrected = torch.argmax(tp)
                generated = torch.cat([generated, corrected.unsqueeze(0)])
                break
        else:
            bonus_probs = target_all_probs[-1]
            bonus_token = torch.multinomial(bonus_probs, num_samples=1).squeeze()
            generated = torch.cat([generated, bonus_token.unsqueeze(0)])

        total_accepted += n_accepted

        if generated.shape[0] - prompt_len >= max_new_tokens:
            break

    acceptance_rate = total_accepted / max(total_draft, 1)
    tokens_per_step = (generated.shape[0] - prompt_len) / max(n_steps, 1)
    speedup = tokens_per_step

    return generated, acceptance_rate, speedup, n_steps


def autoregressive_decode(
    model: nn.Module,
    prompt: torch.Tensor,
    max_new_tokens: int = 20,
    temperature: float = 1.0,
) -> Tuple[torch.Tensor, int]:
    """标准自回归解码（基线对比）

    Args:
        model: 语言模型
        prompt: 输入token序列
        max_new_tokens: 最大生成token数
        temperature: 采样温度

    Returns:
        (generated, n_steps) - 生成序列和前向传播次数
    """
    device = next(model.parameters()).device
    generated = prompt.clone().to(device)
    prompt_len = prompt.shape[0]
    n_steps = 0

    while generated.shape[0] - prompt_len < max_new_tokens:
        n_steps += 1
        with torch.no_grad():
            logits = model(generated.unsqueeze(0))
            probs = F.softmax(logits[0, -1] / max(temperature, 1e-8), dim=-1)
            token = torch.multinomial(probs, num_samples=1).squeeze()
        generated = torch.cat([generated, token.unsqueeze(0)])

    return generated, n_steps


prompt = torch.randint(0, vocab_size, (8,))

result_spec, acc_rate, speedup, steps_spec = speculative_decode(
    draft_model, target_model, prompt, max_new_tokens=20, draft_len=5
)
result_auto, steps_auto = autoregressive_decode(
    target_model, prompt, max_new_tokens=20
)

print(f"=== 投机解码 vs 自回归解码 ===")
print(f"\n{'指标':<20} {'自回归':<15} {'投机解码':<15}")
print("-" * 50)
print(f"{'生成tokens':<20} {result_auto.shape[0]-8:<15} {result_spec.shape[0]-8:<15}")
print(f"{'前向传播次数':<20} {steps_auto:<15} {steps_spec:<15}")
print(f"{'接受率':<20} {'-':<15} {acc_rate:<15.2%}")
print(f"{'每步产出tokens':<20} {1.0:<15.2f} {speedup:<15.2f}")
print(f"\n投机解码核心: 保持与大模型完全相同的输出分布")
print(f"加速取决于: draft模型与target模型的分布匹配度")
print(f"\n关键洞察:")
print(f"  - 自回归: 每步1次前向传播 → 1个token")
print(f"  - 投机解码: 每步1次target前向 + k次draft前向 → 平均{speedup:.1f}个token")
print(f"  - draft模型越小，生成越快；但与target分布差距越大，接受率越低")
print(f"  - 需要在draft速度和接受率之间找到平衡点")

### 自投机解码（Self-Speculative Decoding）

#### 什么是自投机解码？

不使用额外的draft模型，而是利用同一模型的浅层（early exit）来生成候选token，节省draft模型的内存开销。

#### 为什么自投机解码有效？

1. **零额外内存**：不需要加载额外的draft模型，适合端侧设备内存受限的场景。
2. **浅层足够**：对于"简单"token（高频词、标点等），浅层就能给出与深层相似的预测。
3. **分布接近**：同一模型的浅层和深层输出分布天然比不同模型更接近，接受率可能更高。

#### 局限性

浅层与深层分布差异可能较大，接受率通常低于使用独立draft模型的方案。

#### 自投机解码 vs 标准投机解码

| 对比项 | 标准投机解码 | 自投机解码 |
|--------|-------------|------------|
| Draft来源 | 独立小模型 | 同一模型浅层 |
| 额外内存 | 需要加载draft模型 | 无额外内存 |
| 接受率 | 较高（60-80%） | 较低（30-50%） |
| 适用场景 | 服务端（内存充裕） | 端侧（内存紧张） |
| 实现复杂度 | 中等 | 较高（需要修改模型结构） |

In [ ]:
class EarlyExitLM(nn.Module):
    """带早期退出的语言模型，支持自投机解码

    每层都有一个exit head，可以提前输出预测。
    自投机解码时：浅层作为draft，深层作为target。
    """
    def __init__(self, vocab_size: int = 100, dim: int = 64, n_layers: int = 4):
        super().__init__()
        self.vocab_size = vocab_size
        self.n_layers = n_layers
        self.emb = nn.Embedding(vocab_size, dim)
        self.layers = nn.ModuleList([nn.Linear(dim, dim) for _ in range(n_layers)])
        self.exit_heads = nn.ModuleList(
            [nn.Linear(dim, vocab_size) for _ in range(n_layers)]
        )
        self.final_head = nn.Linear(dim, vocab_size)

    def forward(
        self, x: torch.Tensor, exit_layer: Optional[int] = None
    ) -> torch.Tensor:
        """前向传播

        Args:
            x: 输入token序列, shape=(seq_len,)
            exit_layer: 如果指定，在第exit_layer层提前退出
        """
        h = self.emb(x)
        for i, layer in enumerate(self.layers):
            h = F.relu(layer(h))
            if exit_layer is not None and i == exit_layer:
                return self.exit_heads[i](h)
        return self.final_head(h)

model_self_spec = EarlyExitLM(vocab_size=100, dim=64, n_layers=4)
prompt = torch.randint(0, 100, (8,))

with torch.no_grad():
    logits_full = model_self_spec(prompt.unsqueeze(0))
    logits_exit0 = model_self_spec(prompt.unsqueeze(0), exit_layer=0)
    logits_exit1 = model_self_spec(prompt.unsqueeze(0), exit_layer=1)
    logits_exit2 = model_self_spec(prompt.unsqueeze(0), exit_layer=2)

def kl_divergence(p: torch.Tensor, q: torch.Tensor) -> torch.Tensor:
    """计算KL散度 KL(p||q)"""
    return (p * (p.log() - q.log())).sum(dim=-1).mean()

probs_full = F.softmax(logits_full[0, -1], dim=-1)
print(f"=== 各浅层与深层的分布差异 ===")
print(f"(KL散度越小，分布越接近，自投机解码接受率越高)\n")
for name, logits in [
    ("Exit@Layer0", logits_exit0),
    ("Exit@Layer1", logits_exit1),
    ("Exit@Layer2", logits_exit2),
]:
    probs_exit = F.softmax(logits[0, -1], dim=-1)
    kl = kl_divergence(probs_exit, probs_full)
    top1_match = (probs_exit.argmax() == probs_full.argmax()).item()
    print(f"  {name}: KL散度={kl:.4f}, Top-1匹配={top1_match}")

print(f"\n自投机解码策略:")
print(f"  - 用浅层(exit@1)作为draft → 快速生成候选token")
print(f"  - 用深层(全部层)作为target → 验证候选token")
print(f"  - 优势: 无需额外模型，节省内存（端侧关键）")
print(f"  - 代价: 浅层与深层分布差异可能较大，接受率较低")
print(f"  - 实际应用: DeepMind的Self-Speculative Decoding, Medusa等")

---
## 2.3.2 Medusa多头预测

### 什么是Medusa？

Medusa在模型头部添加多个额外的预测头，每个头独立预测未来不同位置的token，一次前向传播生成多个候选，提高投机解码的接受率。无需额外的draft模型。

### 直观理解：Medusa就像"一个脑袋多张嘴"

```
标准LM:  一个头 → 预测下一个token
  [隐藏状态] → LM Head → 下一个token

Medusa:  多个头 → 同时预测多个未来token
  [隐藏状态] → LM Head   → token_{t+1}  (标准头)
           → Head 1   → token_{t+2}  (预测后2步)
           → Head 2   → token_{t+3}  (预测后3步)
           → Head 3   → token_{t+4}  (预测后4步)

  一次前向传播，4个头同时给出4个位置的候选！
  每个头取top-k候选 → 形成树状验证结构
```

### 为什么Medusa有效？

1. **零额外模型**：不需要单独的draft模型，仅增加少量参数（通常<1%总参数）。
2. **一次前向多候选**：每个Medusa头预测未来第$k$个位置的top-k token，形成树状候选结构。
3. **训练成本低**：Medusa头可以冻结基座模型单独训练，几小时即可完成。

### Medusa的数学原理

标准LM头预测下一个token：$P(y_{t+1} | h_t)$

Medusa第$k$个头预测未来第$k+1$个token：
$$P_k(y_{t+k+1} | h_t) = \text{softmax}(W_k h_t + b_k)$$

其中：
- $h_t$：第$t$个位置的最后一层隐藏状态
- $W_k \in \mathbb{R}^{V \times d}$：第$k$个Medusa头的权重矩阵
- $b_k \in \mathbb{R}^{V}$：偏置向量
- $V$：词表大小
- $d$：隐藏维度
- 每个头取top-k候选，形成树状验证结构

### Medusa vs 标准投机解码

| 对比项 | 标准投机解码 | Medusa |
|--------|-------------|--------|
| Draft来源 | 独立小模型 | 同一模型的额外预测头 |
| 额外内存 | 需要加载整个draft模型 | 仅增加<1%参数 |
| 额外训练 | 不需要 | 需要训练Medusa头 |
| 候选结构 | 线性序列 | 树状结构（更多组合） |
| 接受率 | 60-80% | 30-60%（可调） |

In [ ]:
class MedusaHead(nn.Module):
    """Medusa预测头：预测未来第k个位置的token

    每个Medusa头是一个小型MLP，输入为隐藏状态，输出为词表上的logits。
    训练时冻结基座模型，仅训练Medusa头。
    """
    def __init__(self, hidden_dim: int, vocab_size: int, top_k: int = 5):
        super().__init__()
        self.top_k = top_k
        self.head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.SiLU(),
            nn.Linear(hidden_dim // 2, vocab_size),
        )

    def forward(self, hidden_state: torch.Tensor) -> torch.Tensor:
        return self.head(hidden_state)


class MedusaModel(nn.Module):
    """带Medusa头的语言模型

    在标准LM基础上添加多个Medusa头，每个头预测未来不同位置的token。
    生成时：标准头 + Medusa头共同构建树状候选，一次验证。
    """
    def __init__(
        self,
        vocab_size: int = 1000,
        hidden_dim: int = 128,
        n_medusa_heads: int = 3,
        top_k_per_head: int = 5,
    ):
        super().__init__()
        self.vocab_size = vocab_size
        self.hidden_dim = hidden_dim
        self.n_medusa_heads = n_medusa_heads
        self.top_k_per_head = top_k_per_head

        self.embedding = nn.Embedding(vocab_size, hidden_dim)
        self.transformer = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(
                d_model=hidden_dim, nhead=4, batch_first=True
            ),
            num_layers=2,
        )
        self.lm_head = nn.Linear(hidden_dim, vocab_size)
        self.medusa_heads = nn.ModuleList(
            [MedusaHead(hidden_dim, vocab_size, top_k_per_head) for _ in range(n_medusa_heads)]
        )

    def forward(
        self, input_ids: torch.Tensor
    ) -> Tuple[torch.Tensor, List[torch.Tensor]]:
        """前向传播，返回标准logits和各Medusa头的logits"""
        h = self.embedding(input_ids)
        h = self.transformer(h)
        base_logits = self.lm_head(h)
        medusa_logits = [head(h) for head in self.medusa_heads]
        return base_logits, medusa_logits

    @torch.no_grad()
    def generate_candidates(
        self, input_ids: torch.Tensor
    ) -> List[Dict]:
        """生成树状候选结构

        Returns:
            候选路径列表，每条路径包含base_token和medusa_tokens
        """
        self.eval()
        base_logits, medusa_logits = self.forward(input_ids)
        last_pos_logits = base_logits[:, -1, :]
        base_topk = torch.topk(last_pos_logits, self.top_k_per_head, dim=-1)

        candidates: list = []
        for i in range(self.top_k_per_head):
            base_token = base_topk.indices[0, i].item()
            base_prob = base_topk.values[0, i].item()
            medusa_tokens: list = []
            for m_logits in medusa_logits:
                m_top1 = m_logits[0, -1, :].argmax().item()
                medusa_tokens.append(m_top1)
            candidates.append({
                "base_token": base_token,
                "base_prob": base_prob,
                "medusa_tokens": medusa_tokens,
                "path": [base_token] + medusa_tokens,
            })
        return candidates


medusa = MedusaModel(
    vocab_size=1000, hidden_dim=128, n_medusa_heads=3, top_k_per_head=5
)
medusa.eval()
input_ids = torch.randint(0, 1000, (1, 16))

with torch.no_grad():
    base_logits, medusa_logits = medusa(input_ids)
    candidates = medusa.generate_candidates(input_ids)

print(f"=== Medusa多头预测 ===")
print(f"Medusa头数量: {len(medusa.medusa_heads)}")
print(f"每个头Top-K: {medusa.top_k_per_head}")
print(f"候选路径数: {len(candidates)}")
print(f"最大预测长度: 1(base) + {len(medusa.medusa_heads)}(medusa) = {1+len(medusa.medusa_heads)} tokens")
print(f"\n候选路径示例（树状结构）:")
for i, c in enumerate(candidates[:3]):
    print(f"  路径{i}: base={c['base_token']}, medusa={c['medusa_tokens']}, 总长={len(c['path'])} tokens")

base_params = sum(p.numel() for p in medusa.lm_head.parameters())
medusa_params = sum(p.numel() for p in medusa.medusa_heads.parameters())
total_params = sum(p.numel() for p in medusa.parameters())
print(f"\n参数量分析:")
print(f"  LM Head: {base_params:,}")
print(f"  Medusa Heads: {medusa_params:,} ({len(medusa.medusa_heads)}个头)")
print(f"  Medusa占比: {medusa_params/total_params*100:.1f}%")
print(f"  → Medusa头仅增加极少参数，无需额外draft模型")

---
## 2.3.3 连续批处理（Continuous Batching）

### 什么是连续批处理？

不等待整个batch完成后再处理新请求，而是在每个迭代步动态插入新请求、移除已完成请求，显著提升GPU利用率。vLLM等框架的核心调度策略。

### 直观理解：连续批处理就像"流水线"

```
静态批处理（等所有任务完成才接新任务）:
  Batch 1: [任务A(3步), 任务B(5步), 任务C(2步)]
  Step 1: [A✓ B✓ C✓]  ← 3个都在做
  Step 2: [A✓ B✓ C✓]  ← C完成了，但位置空着
  Step 3: [A✓ B✓  --]  ← A也完成了，2个空位
  Step 4: [ -- B✓  --]  ← 只有B在做，GPU利用率25%
  Step 5: [ -- B✓  --]  ← GPU大量空闲！

连续批处理（完成一个立刻补一个）:
  Step 1: [A✓ B✓ C✓]  ← 3个都在做
  Step 2: [A✓ B✓ C✓]  ← C完成，立刻补入D
  Step 3: [A✓ B✓ D✓]  ← A完成，立刻补入E
  Step 4: [E✓ B✓ D✓]  ← 永远满载！
  Step 5: [E✓ B✓ F✓]  ← GPU利用率接近100%
```

### 为什么连续批处理有效？

1. **消除气泡**：静态批处理中，短序列完成后GPU空闲等待长序列，造成大量"气泡"。连续批处理立即用新请求填充空闲位置。
2. **GPU利用率提升**：从静态批处理的30%-50%提升到90%以上。
3. **延迟降低**：新请求无需等待当前batch完成，首token延迟更低。

### 连续批处理的数学分析

设batch大小为$B$，请求$i$的生成长度为$L_i$：
- **静态批处理**：总步数$= \max_i L_i$，GPU利用率$= \frac{\sum_i L_i}{B \cdot \max_i L_i}$
- **连续批处理**：总步数$\approx \frac{\sum_i L_i}{B}$，GPU利用率$\approx 100\%$

当请求长度差异大时，连续批处理的优势更明显。

In [ ]:
class InferenceRequest:
    """推理请求"""
    def __init__(self, req_id: str, max_tokens: int):
        self.req_id = req_id
        self.max_tokens = max_tokens
        self.tokens_generated = 0
        self.done = False

    def generate_one_token(self) -> bool:
        """生成一个token，返回是否完成"""
        self.tokens_generated += 1
        if self.tokens_generated >= self.max_tokens:
            self.done = True
        return self.done


class ContinuousBatchScheduler:
    """连续批处理调度器

    核心思想：每个step动态调度，完成的请求移出，新请求补入。
    这是vLLM等高性能推理框架的核心调度策略。
    """
    def __init__(self, max_batch_size: int = 8):
        self.max_batch_size = max_batch_size
        self.active_requests: Dict[str, InferenceRequest] = {}
        self.request_queue: List[InferenceRequest] = []
        self.completed_requests: List[InferenceRequest] = []
        self.total_tokens_generated = 0
        self.total_steps = 0

    def add_request(self, req_id: str, max_tokens: int = 20) -> None:
        """添加新请求到等待队列"""
        self.request_queue.append(InferenceRequest(req_id, max_tokens))

    def step(self) -> int:
        """执行一步推理调度

        1. 从等待队列中补充请求到active batch
        2. 对active batch中每个请求生成一个token
        3. 移除已完成的请求

        Returns:
            当前active batch大小
        """
        self.total_steps += 1

        while (
            len(self.active_requests) < self.max_batch_size
            and self.request_queue
        ):
            req = self.request_queue.pop(0)
            self.active_requests[req.req_id] = req

        completed_ids: list = []
        for req_id, req in self.active_requests.items():
            req.generate_one_token()
            self.total_tokens_generated += 1
            if req.done:
                completed_ids.append(req_id)

        for req_id in completed_ids:
            self.completed_requests.append(self.active_requests.pop(req_id))

        return len(self.active_requests)

    def is_done(self) -> bool:
        return not self.active_requests and not self.request_queue

    def utilization(self) -> float:
        """当前GPU利用率（active batch占最大batch的比例）"""
        return len(self.active_requests) / self.max_batch_size


class StaticBatchScheduler:
    """静态批处理调度器（对比基线）

    所有请求同时开始，等最长的请求完成后才处理下一批。
    """
    def __init__(self, max_batch_size: int = 8):
        self.max_batch_size = max_batch_size
        self.total_steps = 0
        self.total_tokens_generated = 0

    def run_batch(self, requests: List[InferenceRequest]) -> int:
        """运行一个batch，返回总步数"""
        max_len = max(r.max_tokens for r in requests)
        self.total_steps += max_len
        self.total_tokens_generated += sum(r.max_tokens for r in requests)
        return max_len


np.random.seed(42)
n_requests = 12
request_lengths = [np.random.randint(5, 25) for _ in range(n_requests)]

cont_scheduler = ContinuousBatchScheduler(max_batch_size=4)
for i, length in enumerate(request_lengths):
    cont_scheduler.add_request(f"req_{i}", max_tokens=length)

utilizations: list = []
while not cont_scheduler.is_done():
    cont_scheduler.step()
    utilizations.append(cont_scheduler.utilization())

static_scheduler = StaticBatchScheduler(max_batch_size=4)
for batch_start in range(0, n_requests, 4):
    batch = [InferenceRequest(f"req_{i}", request_lengths[i])
             for i in range(batch_start, min(batch_start + 4, n_requests))]
    static_scheduler.run_batch(batch)

avg_util_cont = np.mean(utilizations)
throughput_cont = cont_scheduler.total_tokens_generated / cont_scheduler.total_steps

print(f"=== 连续批处理 vs 静态批处理 ===")
print(f"总请求数: {n_requests}")
print(f"最大batch: 4")
print(f"请求长度: {request_lengths}")
print(f"\n{'指标':<20} {'静态批处理':<15} {'连续批处理':<15}")
print("-" * 50)
print(f"{'总推理步数':<20} {static_scheduler.total_steps:<15} {cont_scheduler.total_steps:<15}")
print(f"{'总生成tokens':<20} {static_scheduler.total_tokens_generated:<15} {cont_scheduler.total_tokens_generated:<15}")
print(f"{'平均GPU利用率':<20} {'~50%':<15} {avg_util_cont:<15.2%}")
print(f"{'每步吞吐':<20} {static_scheduler.total_tokens_generated/static_scheduler.total_steps:<15.2f} {throughput_cont:<15.2f}")
print(f"\n效率提升: {static_scheduler.total_steps / cont_scheduler.total_steps:.2f}x")
print(f"\n关键洞察:")
print(f"  - 请求长度差异越大，连续批处理优势越明显")
print(f"  - 静态批处理的GPU利用率 = 平均长度/最大长度")
print(f"  - 连续批处理通过动态调度，GPU利用率接近100%")
print(f"  - 产业实践: vLLM默认使用连续批处理")

---
## 2.3.4 早期退出（Early Exit）

### 什么是早期退出？

并非所有token都需要经过全部Transformer层。简单token在浅层即可获得足够置信的输出，可提前退出计算，节省计算量。

### 直观理解：早期退出就像"简单题不用检查"

```
考试做题的类比:

6层Transformer = 6轮检查

简单token（如"的"、"了"、标点）:
  Layer 1: 置信度 0.95 → ✓ 足够确定，直接输出，跳过后5层
  节省 5/6 = 83% 的计算量

中等token（如常见词）:
  Layer 1: 置信度 0.6 → ✗ 不确定
  Layer 2: 置信度 0.7 → ✗ 还不够
  Layer 3: 置信度 0.92 → ✓ 确定，输出
  节省 3/6 = 50% 的计算量

困难token（如需要推理的内容）:
  Layer 1-5: 置信度都不够 → 需要全部6层
  节省 0% 的计算量

平均下来，约50%的token可以在前半层退出
```

### 为什么早期退出有效？

1. **token难度不均**：高频词、标点等"简单"token在浅层就能被正确预测，而需要推理的"困难"token才需要深层计算。
2. **计算量自适应**：平均退出层越浅，计算量节省越多。研究表明约50%的token可以在前半层退出。
3. **精度可控**：通过置信度阈值控制退出条件，阈值越高精度越高但加速越少。

### 早期退出的数学原理

在第$l$层，计算退出置信度：
$$c_l = \max_j P(y | h_l; \theta_l^{exit})$$

退出条件：
$$\text{exit at layer } l \text{ if } c_l > \tau$$

其中：
- $h_l$：第$l$层的隐藏状态
- $\theta_l^{exit}$：第$l$层的退出分类器参数
- $c_l$：第$l$层的预测置信度（最大概率值）
- $\tau$：置信度阈值，控制精度与速度的权衡
- 退出越早，计算量越少：节省$(L - l)/L$的计算量

In [ ]:
class EarlyExitTransformer(nn.Module):
    """带早期退出的Transformer

    每层都有一个exit classifier，当置信度超过阈值时提前退出。
    实际应用中，exit classifier通常是轻量级的线性层。
    """
    def __init__(
        self,
        n_layers: int = 6,
        dim: int = 64,
        n_heads: int = 4,
        n_classes: int = 10,
        threshold: float = 0.9,
    ):
        super().__init__()
        self.n_layers = n_layers
        self.threshold = threshold
        self.layers = nn.ModuleList()
        self.exit_classifiers = nn.ModuleList()
        for _ in range(n_layers):
            self.layers.append(nn.ModuleDict({
                "attn": nn.MultiheadAttention(dim, n_heads, batch_first=True),
                "ffn": nn.Sequential(
                    nn.Linear(dim, dim * 4),
                    nn.GELU(),
                    nn.Linear(dim * 4, dim),
                ),
                "ln1": nn.LayerNorm(dim),
                "ln2": nn.LayerNorm(dim),
            }))
            self.exit_classifiers.append(nn.Linear(dim, n_classes))

    def forward(
        self, x: torch.Tensor, return_exit_info: bool = False
    ) -> Tuple[torch.Tensor, Optional[Dict]]:
        """前向传播，支持早期退出

        Args:
            x: 输入张量, shape=(batch, seq_len, dim)
            return_exit_info: 是否返回退出信息

        Returns:
            (logits, exit_info) 如果return_exit_info=True
            logits 否则
        """
        exit_info: Dict = {"exits": [], "confidences": []}

        for i, layer in enumerate(self.layers):
            h = layer["ln1"](x)
            h, _ = layer["attn"](h, h, h)
            x = x + h
            x = x + layer["ffn"](layer["ln2"](x))

            exit_logits = self.exit_classifiers[i](x.mean(dim=1))
            exit_probs = F.softmax(exit_logits, dim=-1)
            confidence = exit_probs.max(dim=-1).values
            exit_info["exits"].append(i)
            exit_info["confidences"].append(confidence.mean().item())

            if i < self.n_layers - 1 and confidence.min().item() > self.threshold:
                exit_info["final_exit"] = i
                if return_exit_info:
                    return exit_logits, exit_info
                return exit_logits

        exit_info["final_exit"] = self.n_layers - 1
        if return_exit_info:
            return exit_logits, exit_info
        return exit_logits


model_ee = EarlyExitTransformer(
    n_layers=6, dim=64, n_heads=4, n_classes=10, threshold=0.8
)
x = torch.randn(4, 8, 64)

with torch.no_grad():
    out, info = model_ee(x, return_exit_info=True)

print(f"=== 早期退出效果 ===")
print(f"最终退出层: {info['final_exit']}")
print(f"\n各层置信度:")
for layer, conf in zip(info["exits"], info["confidences"]):
    bar = "█" * int(conf * 50)
    marker = " ← 退出" if layer == info["final_exit"] else ""
    print(f"  Layer {layer}: {conf:.4f} {bar}{marker}")

compute_ratio = (info["final_exit"] + 1) / model_ee.n_layers
print(f"\n计算量: 仅使用{info['final_exit']+1}/{model_ee.n_layers}层 ({compute_ratio:.0%})")
print(f"节省: {(1-compute_ratio)*100:.0f}%")

print(f"\n--- 不同阈值下的退出行为 ---")
print(f"{'阈值':<10} {'退出层':<10} {'计算量':<10} {'节省':<10}")
print("-" * 40)
for thresh in [0.5, 0.6, 0.7, 0.8, 0.9, 0.95, 1.0]:
    model_ee.threshold = thresh
    with torch.no_grad():
        _, info_t = model_ee(x, return_exit_info=True)
    ratio = (info_t["final_exit"] + 1) / 6
    print(f"{thresh:<10.2f} {info_t['final_exit']:<10} {ratio:<10.0%} {(1-ratio)*100:<10.0f}%")

model_ee.threshold = 0.8
print(f"\n关键洞察:")
print(f"  - 阈值越低，退出越早，加速越多，但精度可能下降")
print(f"  - 阈值越高，退出越晚，精度越高，但加速越少")
print(f"  - 阈值=1.0意味着永远不提前退出（等价于标准模型）")
print(f"  - 实际应用中，阈值通常设为0.8-0.9，在精度和速度间取得平衡")

---
## 2.3.5 前缀缓存与分块预填充

### 前缀缓存（Prefix Caching）

#### 什么是前缀缓存？

当多个请求共享相同的prompt前缀（如系统提示词、few-shot示例）时，缓存该前缀的KV Cache，后续请求直接复用，避免重复计算。

#### 直观理解：前缀缓存就像"共享的备课笔记"

```
没有前缀缓存:
  用户1: [系统提示 + 问题1] → 计算整个序列的KV Cache
  用户2: [系统提示 + 问题2] → 又计算一遍系统提示的KV Cache！
  用户3: [系统提示 + 问题3] → 又计算一遍！
  → 系统提示被重复计算了3次

有前缀缓存:
  系统提示的KV Cache → 缓存起来
  用户1: [缓存KV + 问题1] → 只计算问题1的KV
  用户2: [缓存KV + 问题2] → 只计算问题2的KV
  用户3: [缓存KV + 问题3] → 只计算问题3的KV
  → 系统提示只计算1次，后续直接复用
```

#### 为什么前缀缓存有效？

1. **系统提示复用**：LLM应用中，系统提示（system prompt）通常很长（数千token）且所有请求共享，缓存后可节省大量prefill计算。
2. **多轮对话加速**：同一用户的多轮对话，前几轮的KV Cache可以直接复用，只需计算最新一轮。
3. **与PagedAttention天然兼容**：vLLM中，前缀对应的block可以被多个sequence共享（copy-on-write）。

#### 前缀缓存的数学分析

设系统提示长度为$L_{prefix}$，用户输入平均长度为$L_{user}$：
- **无缓存**：每个请求的prefill计算量 $\propto (L_{prefix} + L_{user})^2$
- **有缓存**：首个请求prefill计算量 $\propto (L_{prefix} + L_{user})^2$，后续请求 $\propto L_{user}^2$
- **加速比**：当 $L_{prefix} \gg L_{user}$ 时，接近 $(L_{prefix}/L_{user})^2$ 倍

### 分块预填充（Chunked Prefill）

#### 什么是分块预填充？

将长输入的prefill阶段拆分为多个chunk，与decode请求混合调度，避免长prefill请求阻塞短decode请求。

#### 为什么需要分块预填充？

```
没有Chunked Prefill:
  时间线: [====长输入Prefill====][Decode...]
          ↑ 这段时间内，所有decode请求都被阻塞！
          长输入的prefill是计算密集的，可能耗时数百ms

有Chunked Prefill:
  时间线: [Chunk1][D][Chunk2][D][Chunk3][D][Decode...]
          ↑ 长输入被拆成小块，与decode交替执行
          decode请求不会被长时间阻塞
```

1. **避免延迟尖峰**：长输入的prefill可能耗时数百毫秒，期间所有decode请求被阻塞，导致延迟尖峰。
2. **公平调度**：将长prefill拆成chunk后，可以与decode请求交替执行，保证decode的延迟SLA。
3. **更好的GPU利用率**：prefill是计算密集型，decode是内存带宽密集型，混合调度可以更好地利用GPU。

In [ ]:
class PrefixCache:
    """前缀缓存管理器

    缓存共享前缀的KV Cache，避免重复计算。
    实际应用中，vLLM通过PagedAttention的block共享机制实现。
    """
    def __init__(self):
        self._cache: Dict[str, Tuple[torch.Tensor, torch.Tensor]] = {}
        self._hit_count: Dict[str, int] = {}
        self._miss_count = 0

    def _compute_hash(self, token_ids: torch.Tensor) -> str:
        """计算token序列的哈希值作为缓存key"""
        return str(token_ids.tolist())

    def get(
        self, prefix_tokens: torch.Tensor
    ) -> Optional[Tuple[torch.Tensor, torch.Tensor]]:
        """查询缓存

        Args:
            prefix_tokens: 前缀token序列

        Returns:
            缓存的(key, value)张量，如果未命中返回None
        """
        key = self._compute_hash(prefix_tokens)
        if key in self._cache:
            self._hit_count[key] = self._hit_count.get(key, 0) + 1
            return self._cache[key]
        self._miss_count += 1
        return None

    def put(
        self,
        prefix_tokens: torch.Tensor,
        key_cache: torch.Tensor,
        value_cache: torch.Tensor,
    ) -> None:
        """存入缓存"""
        cache_key = self._compute_hash(prefix_tokens)
        self._cache[cache_key] = (key_cache.clone(), value_cache.clone())
        self._hit_count[cache_key] = 0

    def stats(self) -> Dict[str, int]:
        total_hits = sum(self._hit_count.values())
        return {
            "total_hits": total_hits,
            "total_misses": self._miss_count,
            "hit_rate": total_hits / max(total_hits + self._miss_count, 1),
            "cached_prefixes": len(self._cache),
        }


class ChunkedPrefillScheduler:
    """分块预填充调度器

    将长prefill拆分为chunk，与decode请求混合调度。
    避免长prefill阻塞decode请求。
    """
    def __init__(self, chunk_size: int = 64, max_batch_size: int = 8):
        self.chunk_size = chunk_size
        self.max_batch_size = max_batch_size
        self.prefill_chunks: List[Dict] = []
        self.decode_requests: List[Dict] = []
        self.total_steps = 0
        self.total_decode_delays = 0

    def add_prefill_request(
        self, req_id: str, input_length: int
    ) -> None:
        """添加prefill请求，自动拆分为chunk"""
        n_chunks = (input_length + self.chunk_size - 1) // self.chunk_size
        for i in range(n_chunks):
            start = i * self.chunk_size
            end = min(start + self.chunk_size, input_length)
            self.prefill_chunks.append({
                "req_id": req_id,
                "chunk_idx": i,
                "start": start,
                "end": end,
                "is_last": i == n_chunks - 1,
            })

    def add_decode_request(self, req_id: str, max_tokens: int = 20) -> None:
        """添加decode请求"""
        self.decode_requests.append({
            "req_id": req_id,
            "tokens_generated": 0,
            "max_tokens": max_tokens,
            "done": False,
        })

    def step(self) -> Dict[str, int]:
        """执行一步调度：prefill chunk和decode交替执行

        策略：每个step执行1个prefill chunk + 尽可能多的decode
        """
        self.total_steps += 1
        result = {"prefill_chunks": 0, "decode_tokens": 0}

        if self.prefill_chunks:
            chunk = self.prefill_chunks.pop(0)
            result["prefill_chunks"] = 1

        active_decodes = [
            r for r in self.decode_requests if not r["done"]
        ][:self.max_batch_size]

        for req in active_decodes:
            req["tokens_generated"] += 1
            result["decode_tokens"] += 1
            if req["tokens_generated"] >= req["max_tokens"]:
                req["done"] = True

        return result


prefix_cache = PrefixCache()
system_prompt = torch.tensor([1, 2, 3, 4, 5, 6, 7, 8, 9, 10])
mock_k = torch.randn(1, 8, 10, 64)
mock_v = torch.randn(1, 8, 10, 64)

prefix_cache.put(system_prompt, mock_k, mock_v)

print(f"=== 前缀缓存演示 ===")
print(f"缓存系统提示: {system_prompt.tolist()}")

for i in range(5):
    result = prefix_cache.get(system_prompt)
    status = "命中 ✓" if result is not None else "未命中 ✗"
    print(f"  请求{i+1}: {status}")

stats = prefix_cache.stats()
print(f"\n缓存统计: 命中{stats['total_hits']}次, 未命中{stats['total_misses']}次, "
      f"命中率{stats['hit_rate']:.1%}")

print(f"\n=== 分块预填充演示 ===")
chunk_scheduler = ChunkedPrefillScheduler(chunk_size=64, max_batch_size=4)
chunk_scheduler.add_prefill_request("long_input", input_length=256)
chunk_scheduler.add_decode_request("decode_1", max_tokens=10)
chunk_scheduler.add_decode_request("decode_2", max_tokens=15)

print(f"长输入(256 tokens)被拆分为 {len(chunk_scheduler.prefill_chunks)} 个chunk")
print(f"每个chunk大小: {chunk_scheduler.chunk_size} tokens")
print(f"\n调度过程:")
for step in range(20):
    result = chunk_scheduler.step()
    prefill_status = f"Chunk" if result["prefill_chunks"] > 0 else "  --  "
    decode_status = f"{result['decode_tokens']} tokens" if result["decode_tokens"] > 0 else "  --  "
    remaining_chunks = len(chunk_scheduler.prefill_chunks)
    print(f"  Step {step+1:2d}: Prefill={prefill_status}, Decode={decode_status}, "
          f"剩余chunks={remaining_chunks}")
    if remaining_chunks == 0 and all(r["done"] for r in chunk_scheduler.decode_requests):
        break

print(f"\n关键洞察:")
print(f"  - 前缀缓存: 共享prompt只计算1次，后续请求直接复用KV Cache")
print(f"  - 分块预填充: 长输入拆成小块，与decode交替执行，避免阻塞")
print(f"  - 两者结合: vLLM/SGLang等框架的核心优化手段")

---
## 推理加速方法综合对比

不同推理加速方法在加速比、额外内存和输出一致性方面的对比。选择方法时需根据部署场景和资源约束决定。

In [ ]:
print(f"{'方法':<20} {'加速比':<12} {'额外内存':<15} {'输出一致性':<12} {'适用场景':<20}")
print("-" * 79)
methods = [
    ("投机解码", "2-3x", "draft模型", "完全一致", "服务端/端侧(内存充裕)"),
    ("自投机解码", "1.5-2x", "无", "完全一致", "端侧(内存紧张)"),
    ("Medusa多头", "2-3x", "额外预测头(<1%)", "可调", "服务端/端侧"),
    ("连续批处理", "2-4x", "无", "N/A", "高并发服务(必选)"),
    ("早期退出", "1.5-3x", "退出分类器", "近似", "端侧/低延迟场景"),
    ("前缀缓存", "2-10x", "缓存存储", "完全一致", "共享系统提示场景"),
    ("分块预填充", "降低延迟", "无", "完全一致", "长短请求混合"),
]
for name, speedup, mem, consist, scenario in methods:
    print(f"{name:<20} {speedup:<12} {mem:<15} {consist:<12} {scenario:<20}")

print(f"\n=== 产业级选择建议 ===")
print(f"\n1. 服务端部署（GPU服务器）:")
print(f"   - 必选: 连续批处理 + PagedAttention (vLLM默认开启)")
print(f"   - 推荐: 投机解码 (Llama-70B + Llama-7B) 或 Medusa")
print(f"   - 推荐: 前缀缓存 (共享system prompt场景)")
print(f"   - 推荐: 分块预填充 (混合长短请求场景)")
print(f"\n2. 端侧部署（手机/嵌入式，内存紧张）:")
print(f"   - 推荐: 自投机解码 或 早期退出 (无需额外模型)")
print(f"   - 推荐: Medusa (仅增加少量参数)")
print(f"   - 可选: 投机解码 (如果内存允许加载两个模型)")
print(f"\n3. 方法组合:")
print(f"   - 连续批处理 + 投机解码: 最大吞吐")
print(f"   - 连续批处理 + 前缀缓存: 共享prompt场景最优")
print(f"   - 早期退出 + 自投机解码: 端侧最优内存效率")
print(f"\n4. 框架支持:")
print(f"   - vLLM: 连续批处理 + PagedAttention + 投机解码 + 前缀缓存")
print(f"   - SGLang: 连续批处理 + 前缀缓存 + 投机解码")
print(f"   - TGI: 连续批处理 + 投机解码 + 前缀缓存")
print(f"   - llama.cpp: 投机解码 + 前缀缓存 (端侧推荐)")

---
## 产业级实战：真实语言模型投机解码

使用 HuggingFace transformers 加载真实的 GPT-2 模型作为 target model，GPT-2 small 作为 draft model，实现端到端的投机解码，并与标准自回归解码进行性能对比。

> **注意**：此节需要安装 `transformers` 库（`pip install transformers`），且需要网络下载模型权重。如果环境不支持，可跳过此节，前面的小模型演示已覆盖核心逻辑。

In [ ]:
try:
    from transformers import AutoTokenizer, AutoModelForCausalLM
    TRANSFORMERS_AVAILABLE = True
except ImportError:
    TRANSFORMERS_AVAILABLE = False
    print("transformers未安装，跳过产业级实战部分")
    print("安装命令: pip install transformers")

if TRANSFORMERS_AVAILABLE:
    tokenizer = AutoTokenizer.from_pretrained('gpt2')
    target_model = AutoModelForCausalLM.from_pretrained('gpt2-medium')
    draft_model = AutoModelForCausalLM.from_pretrained('gpt2')
    target_model.eval()
    draft_model.eval()

    if torch.cuda.is_available():
        target_model = target_model.cuda()
        draft_model = draft_model.cuda()

    def get_device(model: nn.Module) -> torch.device:
        return next(model.parameters()).device

    def speculative_decode_real(
        target_model: nn.Module,
        draft_model: nn.Module,
        tokenizer: AutoTokenizer,
        prompt: str,
        max_new_tokens: int = 30,
        draft_len: int = 4,
    ) -> Tuple[str, int, int, int, float]:
        """产业级投机解码实现

        Args:
            target_model: 大模型（验证者）
            draft_model: 小模型（草稿生成者）
            tokenizer: 分词器
            prompt: 输入文本
            max_new_tokens: 最大生成token数
            draft_len: 每轮draft生成的候选数

        Returns:
            (generated_text, total_tokens, draft_calls, target_calls, accept_rate)
        """
        device = get_device(target_model)
        input_ids = tokenizer(prompt, return_tensors='pt').input_ids.to(device)
        generated = input_ids
        prompt_len = input_ids.shape[1]
        total_tokens = 0
        draft_calls = 0
        target_calls = 0
        accepted_tokens = 0
        total_draft_tokens = 0

        while total_tokens < max_new_tokens:
            draft_tokens: list = []
            draft_probs_list: list = []
            current = generated

            remaining = max_new_tokens - total_tokens
            k = min(draft_len, remaining)

            with torch.no_grad():
                for _ in range(k):
                    outputs = draft_model(current)
                    next_logits = outputs.logits[:, -1, :]
                    probs = torch.softmax(next_logits, dim=-1)
                    next_token = torch.multinomial(probs, num_samples=1)
                    draft_tokens.append(next_token)
                    draft_probs_list.append(probs)
                    current = torch.cat([current, next_token], dim=-1)
                    draft_calls += 1

            total_draft_tokens += len(draft_tokens)
            all_draft = torch.cat(draft_tokens, dim=-1)
            verify_input = torch.cat([generated, all_draft], dim=-1)

            with torch.no_grad():
                target_outputs = target_model(verify_input)
                target_logits = target_outputs.logits
                target_calls += 1

            n_accepted = 0
            for i in range(len(draft_tokens)):
                pos = generated.shape[1] - 1 + i
                target_prob = torch.softmax(target_logits[0, pos], dim=-1)
                draft_prob = draft_probs_list[i][0]
                token_id = draft_tokens[i][0].item()

                accept_ratio = target_prob[token_id].item() / max(draft_prob[token_id].item(), 1e-10)
                r = torch.rand(1, device=device).item()

                if r < min(accept_ratio, 1.0):
                    n_accepted += 1
                else:
                    diff = torch.clamp(target_prob - draft_prob, min=0)
                    if diff.sum() > 0:
                        corrected = torch.multinomial(
                            diff.unsqueeze(0) / diff.sum(), num_samples=1
                        )
                    else:
                        corrected = target_prob.argmax().unsqueeze(0).unsqueeze(0)
                    draft_tokens[i] = corrected
                    break

            accepted_this_round = n_accepted
            if n_accepted < len(draft_tokens):
                accepted_this_round += 1
            accepted_tokens += accepted_this_round

            new_tokens = torch.cat(
                [draft_tokens[j] for j in range(min(n_accepted + 1, len(draft_tokens)))],
                dim=-1,
            )
            generated = torch.cat([generated, new_tokens], dim=-1)
            total_tokens += new_tokens.shape[-1]

            if n_accepted == len(draft_tokens):
                bonus_logits = target_logits[0, -1]
                bonus_probs = torch.softmax(bonus_logits, dim=-1)
                bonus_token = torch.multinomial(bonus_probs.unsqueeze(0), num_samples=1)
                generated = torch.cat([generated, bonus_token], dim=-1)
                total_tokens += 1
                accepted_tokens += 1

            if total_tokens >= max_new_tokens:
                break

        text = tokenizer.decode(generated[0], skip_special_tokens=True)
        accept_rate = accepted_tokens / max(total_draft_tokens, 1)
        return text, total_tokens, draft_calls, target_calls, accept_rate

    def autoregressive_decode_real(
        model: nn.Module,
        tokenizer: AutoTokenizer,
        prompt: str,
        max_new_tokens: int = 30,
    ) -> Tuple[str, float]:
        """标准自回归解码（基线）"""
        device = get_device(model)
        input_ids = tokenizer(prompt, return_tensors='pt').input_ids.to(device)
        start = time.perf_counter()
        with torch.no_grad():
            output = model.generate(
                input_ids, max_new_tokens=max_new_tokens, do_sample=True,
                temperature=1.0, top_k=0
            )
        elapsed = time.perf_counter() - start
        text = tokenizer.decode(output[0], skip_special_tokens=True)
        return text, elapsed

    prompt = 'The future of artificial intelligence is'

    start = time.perf_counter()
    spec_text, n_tokens, n_draft, n_target, accept_rate = speculative_decode_real(
        target_model, draft_model, tokenizer, prompt,
        max_new_tokens=30, draft_len=4
    )
    spec_time = time.perf_counter() - start

    auto_text, auto_time = autoregressive_decode_real(
        target_model, tokenizer, prompt, max_new_tokens=30
    )

    print(f"=== 产业级投机解码: GPT2-medium(target) + GPT2(draft) ===")
    print(f"\n自回归输出: {auto_text}")
    print(f"投机解码输出: {spec_text}")
    print(f"\n{'指标':<20} {'自回归':<15} {'投机解码':<15}")
    print("-" * 50)
    print(f"{'延迟(s)':<20} {auto_time:<15.3f} {spec_time:<15.3f}")
    print(f"{'target前向次数':<20} {'30':<15} {n_target:<15}")
    print(f"{'draft前向次数':<20} {'0':<15} {n_draft:<15}")
    print(f"{'接受率':<20} {'-':<15} {accept_rate:<15.2%}")

    print(f"\n产业界投机解码实践:")
    print(f"  1. 框架: vLLM (内置speculative decoding)")
    print(f"  2. 模型对: Llama-70B(target) + Llama-7B(draft)")
    print(f"  3. 接受率: 通常60-80% (同系列模型)")
    print(f"  4. 加速比: 1.5-2.5x (取决于接受率和draft_len)")
    print(f"  5. 自投机: Early Exit / Medusa (无需额外draft模型)")

    del target_model, draft_model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

---
## 总结与部署建议

### 推理加速策略全景图

```
推理加速策略
│
├── 减少前向传播次数
│   ├── 投机解码: 小模型生成候选，大模型验证 (2-3x, 输出完全一致)
│   ├── 自投机解码: 浅层生成候选，深层验证 (1.5-2x, 无额外内存)
│   └── Medusa: 多头预测未来token (2-3x, 仅增加<1%参数)
│
├── 提高GPU利用率
│   └── 连续批处理: 动态调度，消除GPU空闲 (2-4x, 高并发必选)
│
├── 减少计算量
│   └── 早期退出: 简单token浅层退出 (1.5-3x, 精度近似)
│
└── 减少重复计算
    ├── 前缀缓存: 复用共享前缀的KV Cache (2-10x, 共享prompt场景)
    └── 分块预填充: 长输入拆块，避免阻塞decode (降低延迟尖峰)
```

### 选择决策流程

```
你的部署场景是什么？
│
├── 服务端（GPU服务器，高并发）
│   → 必选: 连续批处理 + PagedAttention
│   → 加选: 投机解码 / Medusa（提升单请求速度）
│   → 加选: 前缀缓存（有共享system prompt时）
│   → 框架: vLLM / SGLang / TGI
│
├── 端侧（手机/嵌入式，内存紧张）
│   → 推荐: 自投机解码 / 早期退出（无需额外模型）
│   → 可选: Medusa（仅增加少量参数）
│   → 可选: 投机解码（内存允许时）
│   → 框架: llama.cpp / MLC-LLM / ExecuTorch
│
└── 边缘服务器（单GPU，中等并发）
    → 推荐: 连续批处理 + 投机解码
    → 加选: 前缀缓存 + 分块预填充
    → 框架: vLLM / llama.cpp server
```

### 关键要点

1. **投机解码是当前最热门的推理加速方向**：不损失精度，加速比可观，但需要额外的draft模型或修改模型结构。
2. **连续批处理是服务端部署的必选项**：vLLM等框架默认开启，GPU利用率从30%提升到90%+。
3. **端侧优先考虑内存效率**：自投机解码和早期退出不需要额外模型，适合内存受限的场景。
4. **方法可以组合使用**：连续批处理 + 投机解码 + 前缀缓存是服务端的最强组合。
5. **前缀缓存在RAG场景特别有效**：长文档的KV Cache可以跨请求复用，大幅降低延迟。